
# KDMS V4-E — 문헌 근거 기반 피부유형×성분조합 Feature Engineering

기존 V4의 단순 동시 등장 Pair 대신, 논문·공식 과학 의견에서 선정한
**피부유형 조건부 성분 조합 가설**을 Feature로 생성하고 실제 리뷰 Target으로 검증합니다.

## 핵심 원칙

1. 문헌에서 얻은 `+ / -`를 최종 예측값으로 강제하지 않습니다.
2. 문헌은 조합 후보와 **근거 신뢰도**만 제공합니다.
3. 실제 리뷰 위험에 대한 방향과 크기는 모델이 다시 학습합니다.
4. 문헌 규칙과 동일한 개수의 무작위 피부유형 조합을 만들어 대조합니다.
5. 동일 처방 제품이 Train/Test 양쪽에 들어가지 않도록 전성분 집합 기준 Group CV를 사용합니다.

## 생성 파일

1. `v4e_skin_interaction_features.csv`  
   제품×피부유형 단위 모델 입력 및 Target

2. `v4e_rule_dictionary.csv`  
   규칙 정의, 기대 방향, 근거 등급, PMID·SCCS 문서번호

3. `v4e_experiment_results.csv`  
   CatBoost·ElasticNet Ablation, 규칙별 진단, 무작위 규칙 대조 결과


## 1. Google Drive 연결 및 경로 설정

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from collections import defaultdict
from itertools import combinations
import hashlib
import json
import math
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

try:
    from catboost import CatBoostClassifier
except ImportError:
    !pip -q install catboost
    from catboost import CatBoostClassifier

BASE_DIR = Path(
    '/content/drive/MyDrive/0_ASAC_11기_ML_2조/haha'
)

LONG_PATH = (
    BASE_DIR
    / '2_전처리'
    / '전성분long'
    / '전성분_final.csv'
)

V3_PATH = (
    BASE_DIR
    / '3_feature'
    / '서영'
    / 'v3'
    / 'v3_function_features.csv'
)

OUTPUT_DIR = (
    BASE_DIR
    / '3_feature'
    / '서영'
    / '04_evidence'
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 리뷰 파일 경로를 정확히 아는 경우 직접 입력할 수 있습니다.
# None이면 BASE_DIR 아래의 CSV 중 파일명에
# '리뷰', '감성', '최종'이 포함된 파일을 자동 탐색합니다.
REVIEW_PATH = None

# Target 설정
MIN_REVIEWS_PER_PRODUCT_SKIN = 5
HIGH_RISK_THRESHOLD = 0.0833

# 평가 설정
N_SPLITS = 5
N_REPEATS = 3
RANDOM_SEED = 2026

# 문헌규칙 대조실험
RUN_RANDOM_RULE_CONTROL = True
N_RANDOM_RULE_SETS = 30

print("전성분:", LONG_PATH)
print("V3:", V3_PATH)
print("출력:", OUTPUT_DIR)



## 2. 리뷰 파일 자동 탐색 및 데이터 불러오기

리뷰 파일은 다음 컬럼을 사용합니다.

- `상품번호`
- `피부타입`
- `리뷰분류`

현재 리뷰분류는 `긍정`, `부정`으로 가정합니다.


In [ ]:

def normalize_text(value):
    if pd.isna(value):
        return ''
    return unicodedata.normalize('NFKC', str(value)).strip()

def find_review_file(base_dir):
    candidates = []

    for path in base_dir.rglob('*.csv'):
        normalized = normalize_text(path.name)

        if all(token in normalized for token in ['리뷰', '감성', '최종']):
            candidates.append(path)

    if not candidates:
        raise FileNotFoundError(
            "리뷰 감성 최종 CSV를 자동으로 찾지 못했습니다. "
            "REVIEW_PATH에 직접 경로를 입력하세요."
        )

    # 수정 시간이 가장 최근인 파일 우선
    candidates = sorted(
        candidates,
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )

    if len(candidates) > 1:
        print("리뷰 후보 파일:")
        for path in candidates[:10]:
            print("-", path)

    return candidates[0]

if REVIEW_PATH is None:
    REVIEW_PATH = find_review_file(BASE_DIR)
else:
    REVIEW_PATH = Path(REVIEW_PATH)

long_df = pd.read_csv(LONG_PATH, encoding='utf-8-sig')
v3_df = pd.read_csv(V3_PATH, encoding='utf-8-sig')
review_df = pd.read_csv(REVIEW_PATH, encoding='utf-8-sig')

required_long = {'product_id', 'product_name', 'canonical_name'}
required_v3 = {'product_id', 'product_name'}
required_review = {'상품번호', '피부타입', '리뷰분류'}

for name, frame, required in [
    ('전성분', long_df, required_long),
    ('V3', v3_df, required_v3),
    ('리뷰', review_df, required_review),
]:
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"{name} 파일 필수 컬럼 누락: {sorted(missing)}")

print("사용 리뷰 파일:", REVIEW_PATH)
print("전성분 제품 수:", long_df['product_id'].nunique())
print("V3 제품 수:", v3_df['product_id'].nunique())
print("리뷰 수:", len(review_df))
print("리뷰 제품 수:", review_df['상품번호'].nunique())

display(review_df.head())



## 3. 제품 성분 집합 및 피부유형별 리뷰 Target 생성

동일한 전성분 집합을 가진 제품은 같은 `formula_group`으로 묶습니다.

Target:

```text
negative_rate = 부정 리뷰 수 / 전체 리뷰 수
high_risk = negative_rate ≥ 8.33%
```

제품×피부유형 리뷰가 5개 미만인 조합은 기본 실험에서 제외합니다.


In [ ]:

long_df['canonical_name'] = (
    long_df['canonical_name']
    .astype('string')
    .str.strip()
)
long_df = long_df[
    long_df['canonical_name'].notna()
    & long_df['canonical_name'].ne('')
].copy()

product_ingredient_sets = (
    long_df.groupby('product_id')['canonical_name']
    .apply(lambda values: set(values))
    .to_dict()
)

def formula_hash(ingredients):
    formula_text = '\x1f'.join(sorted(ingredients))
    return hashlib.md5(
        formula_text.encode('utf-8')
    ).hexdigest()

formula_group_map = {
    product_id: formula_hash(ingredients)
    for product_id, ingredients in product_ingredient_sets.items()
}

review_df['is_negative'] = (
    review_df['리뷰분류']
    .astype(str)
    .str.strip()
    .eq('부정')
    .astype('int8')
)

target_df = (
    review_df
    .groupby(['상품번호', '피부타입'], as_index=False)
    .agg(
        review_count=('is_negative', 'size'),
        negative_count=('is_negative', 'sum'),
    )
)

target_df['negative_rate'] = (
    target_df['negative_count']
    / target_df['review_count']
)

target_df['high_risk'] = (
    target_df['negative_rate']
    >= HIGH_RISK_THRESHOLD
).astype('int8')

target_df = target_df[
    target_df['review_count']
    >= MIN_REVIEWS_PER_PRODUCT_SKIN
].copy()

model_df = (
    target_df
    .merge(
        v3_df,
        left_on='상품번호',
        right_on='product_id',
        how='inner',
        validate='many_to_one',
    )
)

model_df['formula_group'] = (
    model_df['product_id']
    .map(formula_group_map)
)

model_df = model_df[
    model_df['formula_group'].notna()
].copy()

SKIN_TYPES = sorted(model_df['피부타입'].unique())

print("사용 피부유형:", SKIN_TYPES)
print("제품×피부유형 행:", len(model_df))
print("연결 제품 수:", model_df['product_id'].nunique())
print("전성분 그룹 수:", model_df['formula_group'].nunique())
print("High-risk 비율:", f"{model_df['high_risk'].mean():.2%}")

display(model_df.head())



## 4. 문헌 근거 규칙표

### 근거 가중치 계산

```text
evidence_weight
= 연구설계 신뢰도
× 피부유형·조합 직접성
× 현재 데이터의 크림 제형 적합성
```

이 가중치는 실제 효능 크기가 아닙니다.

- `expected_direction = support`  
  문헌상 피부유형 적합 가능성을 검증할 후보

- `expected_direction = caution`  
  문헌상 불만족·자극 가능성을 검증할 주의 후보

- `evidence_weight = 0`  
  직접 근거가 부족한 탐색 조합으로, 개별 Feature만 생성하고
  문헌 가중 집계 점수에는 포함하지 않음


In [ ]:

RULE_DEFINITIONS = [
    {
        'rule_id': 'dry_humectant_occlusive',
        'skin_types': ['건성', '약건성'],
        'expected_direction': 'support',
        'formula_key': 'dry_humectant_occlusive',
        'study_design_weight': 1.00,
        'directness_weight': 1.00,
        'context_weight': 1.00,
        'source_ids': 'PMID:31532576',
        'evidence_note': (
            '건성 피부에서 glycerol과 petrolatum 조합을 평가한 '
            '무작위 이중맹검 교차시험'
        ),
    },
    {
        'rule_id': 'dry_complete_barrier_lipid',
        'skin_types': ['건성', '약건성'],
        'expected_direction': 'support',
        'formula_key': 'dry_complete_barrier_lipid',
        'study_design_weight': 0.70,
        'directness_weight': 0.80,
        'context_weight': 1.00,
        'source_ids': 'PMID:8618046; PMID:25455137',
        'evidence_note': (
            '세라마이드·콜레스테롤·유리지방산 조성과 '
            '장벽 회복의 관계를 보고한 인체·기전 연구'
        ),
    },
    {
        'rule_id': 'sensitive_fragrance_allergen',
        'skin_types': ['민감성'],
        'expected_direction': 'caution',
        'formula_key': 'sensitive_fragrance_allergen',
        'study_design_weight': 1.00,
        'directness_weight': 0.80,
        'context_weight': 1.00,
        'source_ids': (
            'SCCS/1459/11; SCCS/1589/17; PMID:14572300'
        ),
        'evidence_note': (
            '향료 알레르겐의 피부 감작성과 접촉 알레르기에 대한 '
            'SCCS 의견 및 임상 리뷰'
        ),
    },
    {
        'rule_id': 'sensitive_acid_exfoliant',
        'skin_types': ['민감성'],
        'expected_direction': 'caution',
        'formula_key': 'sensitive_acid_exfoliant',
        'study_design_weight': 0.80,
        'directness_weight': 0.70,
        'context_weight': 0.80,
        'source_ids': 'PMID:20883290; PMID:15725565',
        'evidence_note': (
            '살리실릭애씨드의 감각적·가시적 자극 가능성과 '
            '각질 용해 작용을 평가한 인체 연구'
        ),
    },
    {
        'rule_id': 'sensitive_barrier_soothing',
        'skin_types': ['민감성'],
        'expected_direction': 'support',
        'formula_key': 'sensitive_barrier_soothing',
        'study_design_weight': 0.80,
        'directness_weight': 0.60,
        'context_weight': 1.00,
        'source_ids': 'PMID:16209160; PMID:33984185',
        'evidence_note': (
            '로사시아·습진 대상 장벽 보습제 연구를 민감성 피부 '
            '리뷰에 간접 적용한 조합 가설'
        ),
    },
    {
        'rule_id': 'oily_niacinamide',
        'skin_types': ['지성'],
        'expected_direction': 'support',
        'formula_key': 'oily_niacinamide',
        'study_design_weight': 0.80,
        'directness_weight': 1.00,
        'context_weight': 1.00,
        'source_ids': 'PMID:16766489',
        'evidence_note': (
            '2% 나이아신아마이드의 얼굴 피지 분비를 평가한 '
            '통제 임상시험'
        ),
    },
    {
        'rule_id': 'trouble_niacinamide_ceramide',
        'skin_types': ['트러블성'],
        'expected_direction': 'support',
        'formula_key': 'trouble_niacinamide_ceramide',
        'study_design_weight': 1.00,
        'directness_weight': 1.00,
        'context_weight': 0.70,
        'source_ids': 'PMID:38299457',
        'evidence_note': (
            '여드름 치료제와 함께 사용한 세라마이드·나이아신아마이드 '
            '보습제 무작위 임상시험'
        ),
    },
    {
        'rule_id': 'trouble_acid_with_barrier',
        'skin_types': ['트러블성'],
        'expected_direction': 'support',
        'formula_key': 'trouble_acid_with_barrier',
        'study_design_weight': 0.50,
        'directness_weight': 0.50,
        'context_weight': 0.80,
        'source_ids': (
            'PMID:15725565; PMID:37276158'
        ),
        'evidence_note': (
            '각질 성분과 장벽 관리의 개별 근거를 결합한 '
            '간접 보상형 조합 가설'
        ),
    },
    {
        'rule_id': 'sensitive_fragrance_plus_acid',
        'skin_types': ['민감성'],
        'expected_direction': 'caution',
        'formula_key': 'sensitive_fragrance_plus_acid',
        'study_design_weight': 0.00,
        'directness_weight': 0.00,
        'context_weight': 0.00,
        'source_ids': 'exploratory',
        'evidence_note': (
            '직접 상승작용 근거는 부족하여 문헌 점수에 포함하지 않는 '
            '탐색 조합'
        ),
    },
    {
        'rule_id': 'oily_occlusive_load',
        'skin_types': ['지성'],
        'expected_direction': 'caution',
        'formula_key': 'oily_occlusive_load',
        'study_design_weight': 0.00,
        'directness_weight': 0.00,
        'context_weight': 0.00,
        'source_ids': 'exploratory',
        'evidence_note': (
            '고밀폐 구성이 지성 리뷰에 미치는 영향을 확인하기 위한 '
            '탐색 조합으로, 사전 부정 효과를 가정하지 않음'
        ),
    },
]

rule_df = pd.DataFrame(RULE_DEFINITIONS)

rule_df['evidence_weight'] = (
    rule_df['study_design_weight']
    * rule_df['directness_weight']
    * rule_df['context_weight']
).round(3)

rule_df['skin_types'] = rule_df['skin_types'].map(
    lambda values: '|'.join(values)
)

display(
    rule_df[
        [
            'rule_id',
            'skin_types',
            'expected_direction',
            'evidence_weight',
            'source_ids',
        ]
    ]
)



## 5. 성분군 및 제품별 문헌 규칙 계산

규칙 강도는 함량이 아니라 **전성분 포함 및 기능군 Count 기반 Proxy**입니다.

Count는 과도한 영향이 생기지 않도록 포화함수로 변환합니다.

```text
saturated_count = 1 - exp(-count / scale)
```


In [ ]:

CERAMIDE_NAMES = {
    name
    for name in long_df['canonical_name'].unique()
    if str(name).startswith('세라마이드')
}

CHOLESTEROL_NAMES = {'콜레스테롤'}

FATTY_ACID_NAMES = {
    '스테아릭애씨드',
    '팔미틱애씨드',
    '리놀레익애씨드',
    '리놀레닉애씨드',
}

NIACINAMIDE_NAMES = {'나이아신아마이드'}

FRAGRANCE_ALLERGEN_NAMES = {
    '향료',
    '리모넨',
    '리날룰',
    '시트랄',
    '시트로넬올',
    '제라니올',
    '유제놀',
    '아이소유제놀',
    '쿠마린',
    '파네솔',
    '벤질살리실레이트',
    '벤질벤조에이트',
    '헥실신남알',
    '알파-아이소메틸아이오논',
    '아니스알코올',
}

ACID_EXFOLIANT_NAMES = {
    '살리실릭애씨드',
    '카프릴로일살리실릭애씨드',
    '글라이콜릭애씨드',
    '락틱애씨드',
    '만델릭애씨드',
    '글루코노락톤',
    '락토바이오닉애씨드',
    '아젤라익애씨드',
    '석시닉애씨드',
}

def saturated_count(value, scale):
    value = max(float(value), 0.0)
    return 1.0 - math.exp(-value / scale)

def product_facts(row):
    ingredients = product_ingredient_sets.get(
        row['product_id'],
        set(),
    )

    has_ceramide = bool(ingredients & CERAMIDE_NAMES)
    has_cholesterol = bool(
        ingredients & CHOLESTEROL_NAMES
    )
    has_fatty_acid = bool(
        ingredients & FATTY_ACID_NAMES
    )
    has_niacinamide = bool(
        ingredients & NIACINAMIDE_NAMES
    )

    fragrance_exact_count = len(
        ingredients & FRAGRANCE_ALLERGEN_NAMES
    )
    acid_exact_count = len(
        ingredients & ACID_EXFOLIANT_NAMES
    )

    return {
        'has_ceramide': int(has_ceramide),
        'has_cholesterol': int(has_cholesterol),
        'has_fatty_acid': int(has_fatty_acid),
        'has_niacinamide': int(has_niacinamide),
        'fragrance_exact_count': fragrance_exact_count,
        'acid_exact_count': acid_exact_count,
    }

fact_rows = model_df.apply(
    product_facts,
    axis=1,
    result_type='expand',
)

model_df = pd.concat(
    [
        model_df.reset_index(drop=True),
        fact_rows.reset_index(drop=True),
    ],
    axis=1,
)

def raw_rule_strength(row, formula_key):
    if formula_key == 'dry_humectant_occlusive':
        return math.sqrt(
            saturated_count(
                row['v3_count_humectant'],
                scale=3,
            )
            * saturated_count(
                row['v3_count_occlusive'],
                scale=2,
            )
        )

    if formula_key == 'dry_complete_barrier_lipid':
        return float(
            row['has_ceramide']
            and row['has_cholesterol']
            and row['has_fatty_acid']
        )

    if formula_key == 'sensitive_fragrance_allergen':
        return saturated_count(
            max(
                row['v3_count_fragrance'],
                row['fragrance_exact_count'],
            ),
            scale=1,
        )

    if formula_key == 'sensitive_acid_exfoliant':
        return saturated_count(
            max(
                row['v3_count_acid_exfoliant'],
                row['acid_exact_count'],
            ),
            scale=1,
        )

    if formula_key == 'sensitive_barrier_soothing':
        return math.sqrt(
            saturated_count(
                row['v3_count_barrier'],
                scale=2,
            )
            * saturated_count(
                row['v3_count_soothing'],
                scale=2,
            )
        )

    if formula_key == 'oily_niacinamide':
        return float(row['has_niacinamide'])

    if formula_key == 'trouble_niacinamide_ceramide':
        return float(
            row['has_niacinamide']
            and row['has_ceramide']
        )

    if formula_key == 'trouble_acid_with_barrier':
        return math.sqrt(
            float(
                max(
                    row['v3_count_acid_exfoliant'],
                    row['acid_exact_count'],
                ) > 0
            )
            * saturated_count(
                row['v3_count_barrier'],
                scale=2,
            )
        )

    if formula_key == 'sensitive_fragrance_plus_acid':
        return float(
            max(
                row['v3_count_fragrance'],
                row['fragrance_exact_count'],
            ) > 0
            and max(
                row['v3_count_acid_exfoliant'],
                row['acid_exact_count'],
            ) > 0
        )

    if formula_key == 'oily_occlusive_load':
        return saturated_count(
            row['v3_count_occlusive'],
            scale=3,
        )

    raise KeyError(f"정의되지 않은 formula_key: {formula_key}")

def add_evidence_features(frame, rules, skin_override=None):
    result = frame.copy()

    support_weighted_cols = []
    caution_weighted_cols = []
    support_hit_cols = []
    caution_hit_cols = []

    for rule in rules:
        rule_id = rule['rule_id']
        original_skin_types = rule['skin_types']

        if isinstance(original_skin_types, str):
            original_skin_types = original_skin_types.split('|')

        active_skin_types = (
            skin_override.get(rule_id, original_skin_types)
            if skin_override is not None
            else original_skin_types
        )

        skin_match = (
            result['피부타입']
            .isin(active_skin_types)
            .astype(float)
        )

        raw_strength = result.apply(
            lambda row: raw_rule_strength(
                row,
                rule['formula_key'],
            ),
            axis=1,
        )

        strength_col = f'v4e_rule_{rule_id}_strength'
        hit_col = f'v4e_rule_{rule_id}_hit'
        weighted_col = f'v4e_rule_{rule_id}_weighted'

        result[strength_col] = (
            skin_match * raw_strength
        )
        result[hit_col] = (
            result[strength_col] > 0
        ).astype('int8')
        result[weighted_col] = (
            result[strength_col]
            * float(rule['evidence_weight'])
        )

        if rule['expected_direction'] == 'support':
            support_weighted_cols.append(weighted_col)
            support_hit_cols.append(hit_col)
        elif rule['expected_direction'] == 'caution':
            caution_weighted_cols.append(weighted_col)
            caution_hit_cols.append(hit_col)

    result['v4e_support_score'] = (
        result[support_weighted_cols].sum(axis=1)
    )
    result['v4e_caution_score'] = (
        result[caution_weighted_cols].sum(axis=1)
    )

    # net는 설명·보조 Ablation용입니다.
    # 모델에 방향을 강제하지 않기 위해 support/caution을 별도로 유지합니다.
    result['v4e_net_score'] = (
        result['v4e_support_score']
        - result['v4e_caution_score']
    )

    result['v4e_conflict_score'] = (
        result['v4e_support_score']
        * result['v4e_caution_score']
    )

    result['v4e_support_rule_count'] = (
        result[support_hit_cols].sum(axis=1)
    )
    result['v4e_caution_rule_count'] = (
        result[caution_hit_cols].sum(axis=1)
    )

    return result

rules_for_function = (
    rule_df.to_dict(orient='records')
)

feature_df = add_evidence_features(
    model_df,
    rules_for_function,
)

print("V4-E Feature 생성 완료")
print(
    "규칙 Feature 수:",
    len([
        column
        for column in feature_df.columns
        if column.startswith('v4e_rule_')
    ])
)

display(
    feature_df[
        [
            'product_id',
            '피부타입',
            'review_count',
            'negative_rate',
            'high_risk',
            'v4e_support_score',
            'v4e_caution_score',
            'v4e_net_score',
        ]
    ].head(20)
)



## 6. 규칙별 실제 리뷰 방향 진단

문헌의 기대 방향과 실제 리뷰 데이터의 단순 평균 방향이 같은지 확인합니다.

주의:

- 이 표는 보정되지 않은 단순 비교입니다.
- 제품군·브랜드·리뷰 수 등 교란요인이 있으므로 인과관계로 해석하지 않습니다.
- 문헌의 기대 방향과 리뷰 방향이 다르더라도 규칙을 임의 삭제하지 않고
  Ablation 결과를 함께 확인합니다.


In [ ]:

rule_diagnostic_rows = []

for rule in rules_for_function:
    rule_id = rule['rule_id']
    hit_col = f'v4e_rule_{rule_id}_hit'
    strength_col = f'v4e_rule_{rule_id}_strength'

    active = feature_df[hit_col] == 1
    inactive = feature_df[hit_col] == 0

    active_count = int(active.sum())

    rule_diagnostic_rows.append({
        'section': 'rule_diagnostic',
        'rule_id': rule_id,
        'expected_direction': rule['expected_direction'],
        'evidence_weight': rule['evidence_weight'],
        'active_rows': active_count,
        'active_products': int(
            feature_df.loc[active, 'product_id'].nunique()
        ),
        'active_high_risk_rate': (
            feature_df.loc[active, 'high_risk'].mean()
            if active_count > 0 else np.nan
        ),
        'inactive_high_risk_rate': (
            feature_df.loc[inactive, 'high_risk'].mean()
            if inactive.sum() > 0 else np.nan
        ),
        'active_negative_rate_mean': (
            feature_df.loc[active, 'negative_rate'].mean()
            if active_count > 0 else np.nan
        ),
        'inactive_negative_rate_mean': (
            feature_df.loc[inactive, 'negative_rate'].mean()
            if inactive.sum() > 0 else np.nan
        ),
        'mean_rule_strength': (
            feature_df.loc[active, strength_col].mean()
            if active_count > 0 else np.nan
        ),
    })

rule_diagnostics = pd.DataFrame(
    rule_diagnostic_rows
)

display(rule_diagnostics)



## 7. Ablation 평가 함수

### 비교 모델

- M0: 피부유형만
- M1: 피부유형 + V3 Count
- M2: M1 + 무가중 문헌 규칙 Hit
- M3: M1 + 규칙 강도
- M4: M1 + 근거 가중 개별 규칙
- M5: M1 + 근거 Support/Caution 집계 점수
- M6: M1 + 모든 V4-E 피처

평가 지표:

- ROC-AUC
- PR-AUC
- Recall@Top25%
- Lift@Top25%

동일 처방은 같은 Fold에 배치합니다.


In [ ]:

skin_dummy_df = pd.get_dummies(
    feature_df['피부타입'],
    prefix='skin',
    dtype='int8',
)

experiment_df = pd.concat(
    [
        feature_df.reset_index(drop=True),
        skin_dummy_df.reset_index(drop=True),
    ],
    axis=1,
)

skin_columns = list(skin_dummy_df.columns)

v3_count_columns = [
    column
    for column in experiment_df.columns
    if column.startswith('v3_count_')
]

rule_hit_columns = [
    column
    for column in experiment_df.columns
    if column.startswith('v4e_rule_')
    and column.endswith('_hit')
]

rule_strength_columns = [
    column
    for column in experiment_df.columns
    if column.startswith('v4e_rule_')
    and column.endswith('_strength')
]

rule_weighted_columns = [
    column
    for column in experiment_df.columns
    if column.startswith('v4e_rule_')
    and column.endswith('_weighted')
]

aggregate_score_columns = [
    'v4e_support_score',
    'v4e_caution_score',
    'v4e_conflict_score',
    'v4e_support_rule_count',
    'v4e_caution_rule_count',
]

feature_sets = {
    'M0_skin_only': skin_columns,
    'M1_skin_v3_count': (
        skin_columns + v3_count_columns
    ),
    'M2_v3_rule_hits': (
        skin_columns
        + v3_count_columns
        + rule_hit_columns
    ),
    'M3_v3_rule_strength': (
        skin_columns
        + v3_count_columns
        + rule_strength_columns
    ),
    'M4_v3_evidence_weighted': (
        skin_columns
        + v3_count_columns
        + rule_weighted_columns
    ),
    'M5_v3_evidence_scores': (
        skin_columns
        + v3_count_columns
        + aggregate_score_columns
    ),
    'M6_v3_evidence_all': (
        skin_columns
        + v3_count_columns
        + rule_hit_columns
        + rule_strength_columns
        + rule_weighted_columns
        + aggregate_score_columns
    ),
}

y = experiment_df['high_risk'].to_numpy()
groups = experiment_df['formula_group'].to_numpy()

def top_quartile_metrics(y_true, probabilities):
    n_top = max(
        1,
        int(math.ceil(len(y_true) * 0.25)),
    )

    top_indices = np.argsort(
        -probabilities
    )[:n_top]

    positive_total = max(
        int(y_true.sum()),
        1,
    )

    recall_top25 = (
        y_true[top_indices].sum()
        / positive_total
    )

    baseline_rate = y_true.mean()
    top_rate = y_true[top_indices].mean()

    lift_top25 = (
        top_rate / baseline_rate
        if baseline_rate > 0 else np.nan
    )

    return recall_top25, lift_top25

def build_model(model_name, seed):
    if model_name == 'catboost':
        return CatBoostClassifier(
            iterations=300,
            depth=4,
            learning_rate=0.03,
            loss_function='Logloss',
            eval_metric='AUC',
            auto_class_weights='Balanced',
            l2_leaf_reg=5,
            random_seed=seed,
            verbose=False,
            allow_writing_files=False,
        )

    if model_name == 'elasticnet':
        return make_pipeline(
            StandardScaler(),
            LogisticRegression(
                penalty='elasticnet',
                solver='saga',
                l1_ratio=0.5,
                C=0.5,
                class_weight='balanced',
                max_iter=5000,
                random_state=seed,
            ),
        )

    raise ValueError(model_name)

def evaluate_feature_sets(
    data,
    targets,
    group_values,
    feature_set_dict,
    model_names=('catboost', 'elasticnet'),
    n_repeats=3,
):
    fold_rows = []

    for repeat in range(n_repeats):
        seed = RANDOM_SEED + repeat

        splitter = StratifiedGroupKFold(
            n_splits=N_SPLITS,
            shuffle=True,
            random_state=seed,
        )

        for fold, (train_index, test_index) in enumerate(
            splitter.split(
                data,
                targets,
                group_values,
            )
        ):
            y_train = targets[train_index]
            y_test = targets[test_index]

            for model_name in model_names:
                for feature_set_name, columns in feature_set_dict.items():
                    model = build_model(
                        model_name,
                        seed,
                    )

                    model.fit(
                        data.iloc[train_index][columns],
                        y_train,
                    )

                    probabilities = model.predict_proba(
                        data.iloc[test_index][columns]
                    )[:, 1]

                    recall_top25, lift_top25 = (
                        top_quartile_metrics(
                            y_test,
                            probabilities,
                        )
                    )

                    fold_rows.append({
                        'section': 'cv_fold',
                        'model': model_name,
                        'feature_set': feature_set_name,
                        'repeat': repeat,
                        'fold': fold,
                        'roc_auc': roc_auc_score(
                            y_test,
                            probabilities,
                        ),
                        'pr_auc': average_precision_score(
                            y_test,
                            probabilities,
                        ),
                        'recall_top25': recall_top25,
                        'lift_top25': lift_top25,
                        'n_train': len(train_index),
                        'n_test': len(test_index),
                    })

    return pd.DataFrame(fold_rows)

cv_results = evaluate_feature_sets(
    experiment_df,
    y,
    groups,
    feature_sets,
    n_repeats=N_REPEATS,
)

cv_summary = (
    cv_results
    .groupby(
        ['model', 'feature_set'],
        as_index=False,
    )
    .agg(
        roc_auc_mean=('roc_auc', 'mean'),
        roc_auc_std=('roc_auc', 'std'),
        pr_auc_mean=('pr_auc', 'mean'),
        pr_auc_std=('pr_auc', 'std'),
        recall_top25_mean=('recall_top25', 'mean'),
        recall_top25_std=('recall_top25', 'std'),
        lift_top25_mean=('lift_top25', 'mean'),
        lift_top25_std=('lift_top25', 'std'),
        n_evaluations=('pr_auc', 'size'),
    )
)

cv_summary.insert(0, 'section', 'cv_summary')

display(
    cv_summary.sort_values(
        ['model', 'pr_auc_mean'],
        ascending=[True, False],
    )
)



## 8. 무작위 피부유형 규칙 대조군

문헌 규칙의 성능 향상이 단순히 Interaction Feature를 추가했기 때문인지 확인합니다.

각 규칙의:

- 성분 조건
- 근거 가중치
- Support/Caution 구분
- 대상 피부유형 개수

는 유지하고, 피부유형만 무작위로 다시 배정합니다.

ElasticNet에서 문헌 규칙의 PR-AUC와 무작위 규칙 분포를 비교합니다.


In [ ]:

random_control_rows = []

def random_skin_override(rule_records, skin_types, rng):
    override = {}

    for rule in rule_records:
        rule_id = rule['rule_id']

        original = rule['skin_types']
        if isinstance(original, str):
            original = original.split('|')

        n_skin = len(original)

        chosen = list(
            rng.choice(
                skin_types,
                size=n_skin,
                replace=False,
            )
        )

        # 가능한 경우 원래와 완전히 같은 선택을 피함
        if (
            set(chosen) == set(original)
            and len(skin_types) > n_skin
        ):
            for _ in range(20):
                chosen = list(
                    rng.choice(
                        skin_types,
                        size=n_skin,
                        replace=False,
                    )
                )
                if set(chosen) != set(original):
                    break

        override[rule_id] = chosen

    return override

def make_random_experiment_frame(
    base_feature_df,
    rule_records,
    skin_types,
    seed,
):
    rng = np.random.default_rng(seed)

    override = random_skin_override(
        rule_records,
        skin_types,
        rng,
    )

    random_feature_df = add_evidence_features(
        model_df,
        rule_records,
        skin_override=override,
    )

    random_skin_dummies = pd.get_dummies(
        random_feature_df['피부타입'],
        prefix='skin',
        dtype='int8',
    )

    random_experiment_df = pd.concat(
        [
            random_feature_df.reset_index(drop=True),
            random_skin_dummies.reset_index(drop=True),
        ],
        axis=1,
    )

    return random_experiment_df, override

if RUN_RANDOM_RULE_CONTROL:
    observed_feature_set = {
        'observed_evidence': (
            skin_columns
            + v3_count_columns
            + rule_weighted_columns
            + aggregate_score_columns
        )
    }

    # 동일한 고정 Split로 observed/random 비교
    fixed_splitter = StratifiedGroupKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_SEED,
    )

    fixed_folds = list(
        fixed_splitter.split(
            experiment_df,
            y,
            groups,
        )
    )

    def fixed_fold_pr_auc(data, columns):
        fold_scores = []

        for fold, (train_index, test_index) in enumerate(
            fixed_folds
        ):
            model = build_model(
                'elasticnet',
                RANDOM_SEED + fold,
            )

            model.fit(
                data.iloc[train_index][columns],
                y[train_index],
            )

            probabilities = model.predict_proba(
                data.iloc[test_index][columns]
            )[:, 1]

            fold_scores.append(
                average_precision_score(
                    y[test_index],
                    probabilities,
                )
            )

        return float(np.mean(fold_scores))

    observed_pr_auc = fixed_fold_pr_auc(
        experiment_df,
        observed_feature_set['observed_evidence'],
    )

    for random_index in range(N_RANDOM_RULE_SETS):
        random_frame, override = (
            make_random_experiment_frame(
                feature_df,
                rules_for_function,
                SKIN_TYPES,
                RANDOM_SEED + 1000 + random_index,
            )
        )

        random_rule_weighted_columns = [
            column
            for column in random_frame.columns
            if column.startswith('v4e_rule_')
            and column.endswith('_weighted')
        ]

        random_columns = (
            skin_columns
            + v3_count_columns
            + random_rule_weighted_columns
            + aggregate_score_columns
        )

        random_pr_auc = fixed_fold_pr_auc(
            random_frame,
            random_columns,
        )

        random_control_rows.append({
            'section': 'random_rule_control',
            'random_index': random_index,
            'observed_pr_auc': observed_pr_auc,
            'random_pr_auc': random_pr_auc,
            'delta_observed_minus_random': (
                observed_pr_auc - random_pr_auc
            ),
            'skin_override_json': json.dumps(
                override,
                ensure_ascii=False,
                sort_keys=True,
            ),
        })

    random_control_df = pd.DataFrame(
        random_control_rows
    )

    empirical_p = (
        1
        + (
            random_control_df['random_pr_auc']
            >= observed_pr_auc
        ).sum()
    ) / (1 + len(random_control_df))

    random_summary_df = pd.DataFrame([{
        'section': 'random_rule_summary',
        'observed_pr_auc': observed_pr_auc,
        'random_pr_auc_mean': (
            random_control_df['random_pr_auc'].mean()
        ),
        'random_pr_auc_std': (
            random_control_df['random_pr_auc'].std()
        ),
        'random_pr_auc_max': (
            random_control_df['random_pr_auc'].max()
        ),
        'observed_percentile_vs_random': (
            random_control_df['random_pr_auc']
            .lt(observed_pr_auc)
            .mean()
        ),
        'empirical_p_value': empirical_p,
        'n_random_sets': len(random_control_df),
    }])

    display(random_summary_df)
else:
    random_control_df = pd.DataFrame()
    random_summary_df = pd.DataFrame()
    print("무작위 규칙 대조실험을 건너뜁니다.")



## 9. 결과 저장

모델 입력에서는 다음처럼 선택할 수 있습니다.

### V3 기본

```python
v3_count_cols = [
    c for c in df.columns
    if c.startswith('v3_count_')
]
```

### 문헌 규칙 개별값

```python
weighted_rule_cols = [
    c for c in df.columns
    if c.startswith('v4e_rule_')
    and c.endswith('_weighted')
]
```

### 문헌 집계 점수

```python
score_cols = [
    'v4e_support_score',
    'v4e_caution_score',
    'v4e_conflict_score',
]
```


In [ ]:

FEATURE_OUTPUT_PATH = (
    OUTPUT_DIR
    / 'v4e_skin_interaction_features.csv'
)

RULE_OUTPUT_PATH = (
    OUTPUT_DIR
    / 'v4e_rule_dictionary.csv'
)

EXPERIMENT_OUTPUT_PATH = (
    OUTPUT_DIR
    / 'v4e_experiment_results.csv'
)

feature_output_columns = [
    'product_id',
    'product_name',
    '피부타입',
    'review_count',
    'negative_count',
    'negative_rate',
    'high_risk',
    'formula_group',
] + [
    column
    for column in feature_df.columns
    if column.startswith('v3_count_')
    or column.startswith('v4e_')
]

feature_output = feature_df[
    feature_output_columns
].copy()

# 실험 결과를 section 컬럼으로 하나의 CSV에 통합
experiment_parts = [
    cv_summary,
    cv_results,
    rule_diagnostics,
]

if not random_summary_df.empty:
    experiment_parts.append(random_summary_df)

if not random_control_df.empty:
    experiment_parts.append(random_control_df)

experiment_output = pd.concat(
    experiment_parts,
    ignore_index=True,
    sort=False,
)

feature_output.to_csv(
    FEATURE_OUTPUT_PATH,
    index=False,
    encoding='utf-8-sig',
)

rule_df.to_csv(
    RULE_OUTPUT_PATH,
    index=False,
    encoding='utf-8-sig',
)

experiment_output.to_csv(
    EXPERIMENT_OUTPUT_PATH,
    index=False,
    encoding='utf-8-sig',
)

# 기본 검증
assert feature_output[
    ['product_id', '피부타입']
].duplicated().sum() == 0

assert feature_output.isna().sum().sum() == 0
assert feature_output['high_risk'].isin([0, 1]).all()

print("저장 완료")
print("1.", FEATURE_OUTPUT_PATH)
print("2.", RULE_OUTPUT_PATH)
print("3.", EXPERIMENT_OUTPUT_PATH)



## 10. 해석 기준

### 채택 기준

문헌 규칙은 다음 조건을 모두 만족할 때 최종 V4로 채택합니다.

1. V3 Count 대비 PR-AUC 또는 Lift@25%가 반복 CV에서 개선
2. 개선 방향이 최소 리뷰 수 5·10 조건에서 유지
3. 문헌 규칙이 무작위 피부유형 규칙보다 우수
4. 동일 처방 Group Split에서도 개선 유지
5. 규칙이 소수 제품·특정 브랜드만 식별하지 않음

### 결과 표현

가능:

> 민감성 피부와 향료 알레르겐 조합 피처가 모델의
> High-risk 스크리닝에 추가 정보를 제공하였다.

금지:

> 향료가 민감성 피부의 부정 반응을 유발한다.

본 연구의 결과는 리뷰 기반 연관성 및 스크리닝 결과이며,
성분의 임상적 효능·안전성 또는 인과관계를 판정하지 않습니다.
